# Grounded Document Assistant
An AI application that answers questions using information retrieved from uploaded documents, provides source references, and acknowledges when the available evidence is insufficient.

# Project Setup

In [ ]:
from pathlib import Path

print("Project setup started!")

Project setup started!


In [ ]:
# Get the current working directory
project_path = Path.cwd()

print("Current working directory:", project_path)

Current working directory: /content


In [ ]:
data_path = project_path/"data"
data_path.mkdir(exist_ok = True)

print("Folder Available : ",data_path.exists())
print("Data Path :",data_path)

Folder Available :  True
Data Path : /content/data


# Data Import

In [ ]:
pdf_path = data_path/"deep_ocean_resource_guide.pdf"

# Downloading the pdf from the NOAA Repository
!wget -O /content/data/deep_ocean_resource_guide.pdf https://repository.library.noaa.gov/view/noaa/36065/noaa_36065_DS1.pdf

if pdf_path.exists():
  print("Pdf Existed")
  print("Pdf Size : ", round(pdf_path.stat().st_size/(1024**2), 2), "MB")


--2026-11-02 11:39:16--  https://repository.library.noaa.gov/view/noaa/36065/noaa_36065_DS1.pdf
Resolving repository.library.noaa.gov (repository.library.noaa.gov)... 23.59.88.241, 23.59.88.235, 2600:1409:9800:1d::17d8:9106, ...
Connecting to repository.library.noaa.gov (repository.library.noaa.gov)|23.59.88.241|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 971112 (948K) [application/pdf]
Saving to: ‘/content/data/deep_ocean_resource_guide.pdf’

/content/data/deep_ 100%[===================>] 948.35K  --.-KB/s    in 0.08s   

2026-11-02 11:39:17 (12.2 MB/s) - ‘/content/data/deep_ocean_resource_guide.pdf’ saved [971112/971112]

Pdf Existed
Pdf Size :  0.93 MB


# Installing pypdf
It's a Python library for reading PDF files and extracting their text and metadata. We're using it instead of manually opening and copying the document's contents

In [ ]:
%pip install -q pypdf
print("pypdf successfully installed")

pypdf successfully installed


# Exploring the PDF

In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

# number of pages in pdf
print("Number of Pages : ", len(reader.pages))

# Extracting first page content and printing
first_page_text = reader.pages[0].extract_text()

print("\nFirst Page Text:\n",first_page_text)

Number of Pages :  8

First Page Text:
 O
n November 30,
a research team
led by University of
Delaware marine 
scientist Craig Cary
will set sail from
Manzanillo, Mexico,
on the 21-day expedi-
tion Extreme 2004:
Exploring the 
Deep Frontier.
They will live aboard the 274-foot
research vessel Atlantis and dive to
the depths in the submersible Alvin.
Their mission will be to explore
one of the most demanding environ-
ments on the planet: hydrothermal
vents.These underwater geysers
erupt at cracks in the Earth’s crust,
where new seafloor is being born.
The scientists will be working 
to learn more about biocomplexity,
the complex interplay between vent
organisms and their environment.
Biologists, chemists, and geologists
will all be working together to better
understand how the strange life 
at the vents can function in their 
hostile environment of extreme 
temperature, toxic chemicals, high
pressure, and darkness.
Using tools borrowed from the
Human Genome Project, the scien-
tists will

In [ ]:
pages_text = []

for page_number, page in enumerate(reader.pages, start = 1):
  page_text = page.extract_text() or ""
  pages_text.append(page_text)

  print(f'Page {page_number} : {len(page_text)} Characters', )

Page 1 : 3721 Characters
Page 2 : 5179 Characters
Page 3 : 4817 Characters
Page 4 : 5387 Characters
Page 5 : 4913 Characters
Page 6 : 6336 Characters
Page 7 : 4250 Characters
Page 8 : 5459 Characters


# Text Cleaning

In [ ]:
# Inspecting the text quality

print(pages_text[1][:1500])

Geology of the Ocean Floor
Did you know that the Earth’s longest mountain range is underwater? The Mid-Ocean Ridge system, shown above
snaking its way between the continents, is more than 56,000 kilometers (35,000 mi) long. It marks the areas where the
Earth’s crustal plates are moving apart. This is where most hydrothermal vents are located. The Extreme 2004 dive
site, marked above by the submersibleAlvin, is in an active vent field in the Pacific Ocean west of Costa Rica.
M
mountains. In fact, the Earth’s
longest mountain range is under-
water. Over 56,000 kilometers
(35,000 mi) long, this mountain
range, called the Mid-Ocean Ridge
system, snakes around the globe.
The Mid-Ocean Ridge marks
one of the most geologically active
areas on Earth. It is where the
planet’s crustal
plates are moving
apart. It is where
new seafloor is
being born, giving
rise to hydrothermal
vents and volcanoes.
Adapted from NOAA
T
o someone who can’t swim,
the “deep sea” may be just
above ankle level in the oc

In [ ]:
import re

for page_number, text in enumerate(pages_text, start=1):
    single_letter_lines = re.findall(
        r"(?m)^\s*[A-Za-z]\s*$",
        text
    )

    hyphenated_line_breaks = re.findall(
        r"-\s*\n\s*[a-z]",
        text
    )

    print(f"Page {page_number}")
    print("  Single-letter lines:", single_letter_lines)
    print("  Possible split words:", len(hyphenated_line_breaks))

Page 1
  Single-letter lines: ['O', 'T', 'M', 'M', 'M']
  Possible split words: 9
Page 2
  Single-letter lines: ['M', 'T', 'U', 'M', 'O']
  Possible split words: 10
Page 3
  Single-letter lines: ['M', 'M', 'M']
  Possible split words: 7
Page 4
  Single-letter lines: ['T', 'M', 'I', 'M', 'M']
  Possible split words: 10
Page 5
  Single-letter lines: ['I', 'M', 'A', 'M']
  Possible split words: 15
Page 6
  Single-letter lines: ['T', 'M', 'M', 'B']
  Possible split words: 20
Page 7
  Single-letter lines: ['M', 'M', 'M  ']
  Possible split words: 10
Page 8
  Single-letter lines: ['T', 'M']
  Possible split words: 3


In [ ]:
import re

for page_number, text in enumerate(pages_text, start=1):
    matches = list(re.finditer(r"-\s*\n\s*[a-z]", text))

    if matches:
        print(f"\n--- Page {page_number} ---")

        for match in matches[:3]:
            start = max(0, match.start() - 45)
            end = min(len(text), match.end() + 45)

            print(repr(text[start:end]))



--- Page 1 ---
'from\nManzanillo, Mexico,\non the 21-day expedi-\ntion Extreme 2004:\nExploring the \nDeep Frontie'
' to explore\none of the most demanding environ-\nments on the planet: hydrothermal\nvents.These '
'owed from the\nHuman Genome Project, the scien-\ntists will examine creatures such as\nthe Pompe'

--- Page 2 ---
', the Earth’s\nlongest mountain range is under-\nwater. Over 56,000 kilometers\n(35,000 mi) long'
'm each of the\nmajor land masses. It’s the sub-\nmerged portion of the continents.\nThe shelf ha'
'ry when they descended in the U.S. Navy bathy-\nscaphe Trieste (left) to the bottom of the\nMar'

--- Page 3 ---
'ess is\ncalled subduction. It results in earth-\nquakes, volcanoes, and the formation of\ndeep o'
'\nthat make up the Earth’s crust and the direc-\ntions in which they are moving.\nM\nAdapted from'
'understand Earth’s\nstructure, think of a hard-\nboiled egg. The crust is like the\negg’s thin s'

--- Page 4 ---
'ermal vent sites!\nThese creatures, from m

### Checking "Clean Hyphenated Line Breaks"

In [ ]:
import re

def clean_hyphenated_line_breaks(text):
    cleaned_text = re.sub(r"-\s*\n\s*([a-z])", r"\1", text)
    return cleaned_text

cleaned_page_1 = clean_hyphenated_line_breaks(pages_text[0])

print("BEFORE:")
print(pages_text[0][:500])

print("\nAFTER:")
print(cleaned_page_1[:500])


BEFORE:
O
n November 30,
a research team
led by University of
Delaware marine 
scientist Craig Cary
will set sail from
Manzanillo, Mexico,
on the 21-day expedi-
tion Extreme 2004:
Exploring the 
Deep Frontier.
They will live aboard the 274-foot
research vessel Atlantis and dive to
the depths in the submersible Alvin.
Their mission will be to explore
one of the most demanding environ-
ments on the planet: hydrothermal
vents.These underwater geysers
erupt at cracks in the Earth’s crust,
where new seafloor

AFTER:
O
n November 30,
a research team
led by University of
Delaware marine 
scientist Craig Cary
will set sail from
Manzanillo, Mexico,
on the 21-day expedition Extreme 2004:
Exploring the 
Deep Frontier.
They will live aboard the 274-foot
research vessel Atlantis and dive to
the depths in the submersible Alvin.
Their mission will be to explore
one of the most demanding environments on the planet: hydrothermal
vents.These underwater geysers
erupt at cracks in the Earth’s crust,
where

### Checking "Clean Split Words"

In [ ]:
def clean_split_words(text):
    return re.sub(
        r"(?m)^([A-Z])\n([a-z])",
        r"\1\2",
        text
    )

    return cleaned_text

cleaned_page_1 = clean_hyphenated_line_breaks(pages_text[0])
cleaned_page_1 = clean_split_words(cleaned_page_1)

print(cleaned_page_1[:250])


On November 30,
a research team
led by University of
Delaware marine 
scientist Craig Cary
will set sail from
Manzanillo, Mexico,
on the 21-day expedition Extreme 2004:
Exploring the 
Deep Frontier.
They will live aboard the 274-foot
research vessel 


### Checking "clean SPlit Words"

In [ ]:
test_page = clean_split_words(pages_text[0])

position = test_page.find("ocean’s greatest")
print(repr(test_page[position - 10:position + 40]))


'e2004\nThe ocean’s greatest\ndepths once were\nbeliev'


### Checking "Clean Missing Spaces"

In [ ]:
def clean_missing_spaces(text):
    return re.sub(r"([.!?])([A-Z])", r"\1 \2", text)

cleaned_page_1 = clean_hyphenated_line_breaks(pages_text[0])
cleaned_page_1 = clean_split_words(cleaned_page_1)
cleaned_page_1 = clean_missing_spaces(cleaned_page_1)

print(cleaned_page_1[:1000])

On November 30,
a research team
led by University of
Delaware marine 
scientist Craig Cary
will set sail from
Manzanillo, Mexico,
on the 21-day expedition Extreme 2004:
Exploring the 
Deep Frontier.
They will live aboard the 274-foot
research vessel Atlantis and dive to
the depths in the submersible Alvin.
Their mission will be to explore
one of the most demanding environments on the planet: hydrothermal
vents. These underwater geysers
erupt at cracks in the Earth’s crust,
where new seafloor is being born.
The scientists will be working 
to learn more about biocomplexity,
the complex interplay between vent
organisms and their environment.
Biologists, chemists, and geologists
will all be working together to better
understand how the strange life 
at the vents can function in their 
hostile environment of extreme 
temperature, toxic chemicals, high
pressure, and darkness.
Using tools borrowed from the
Human Genome Project, the scientists will examine creatures such as
the Pompeii worm, w

### Checking "Extra Characters"

In [ ]:
print(repr(pages_text[0][:1500]))

'O\nn November 30,\na research team\nled by University of\nDelaware marine \nscientist Craig Cary\nwill set sail from\nManzanillo, Mexico,\non the 21-day expedi-\ntion Extreme 2004:\nExploring the \nDeep Frontier.\nThey will live aboard the 274-foot\nresearch vessel Atlantis and dive to\nthe depths in the submersible Alvin.\nTheir mission will be to explore\none of the most demanding environ-\nments on the planet: hydrothermal\nvents.These underwater geysers\nerupt at cracks in the Earth’s crust,\nwhere new seafloor is being born.\nThe scientists will be working \nto learn more about biocomplexity,\nthe complex interplay between vent\norganisms and their environment.\nBiologists, chemists, and geologists\nwill all be working together to better\nunderstand how the strange life \nat the vents can function in their \nhostile environment of extreme \ntemperature, toxic chemicals, high\npressure, and darkness.\nUsing tools borrowed from the\nHuman Genome Project, the scien-\ntists will exam

In [ ]:
def normalize_line_breaks(text):
    # Replace one or more line breaks with a space
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # Reduce three or more consecutive newlines to two
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove extra spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


cleaned_page_1 = clean_hyphenated_line_breaks(pages_text[0])
cleaned_page_1 = clean_split_words(cleaned_page_1)
cleaned_page_1 = clean_missing_spaces(cleaned_page_1)
normalized_page_1 = normalize_line_breaks(cleaned_page_1)

print(normalized_page_1[:1500])


On November 30, a research team led by University of Delaware marine scientist Craig Cary will set sail from Manzanillo, Mexico, on the 21-day expedition Extreme 2004: Exploring the Deep Frontier. They will live aboard the 274-foot research vessel Atlantis and dive to the depths in the submersible Alvin. Their mission will be to explore one of the most demanding environments on the planet: hydrothermal vents. These underwater geysers erupt at cracks in the Earth’s crust, where new seafloor is being born. The scientists will be working to learn more about biocomplexity, the complex interplay between vent organisms and their environment. Biologists, chemists, and geologists will all be working together to better understand how the strange life at the vents can function in their hostile environment of extreme temperature, toxic chemicals, high pressure, and darkness. Using tools borrowed from the Human Genome Project, the scientists will examine creatures such as the Pompeii worm, which c

# Checking Cleaning Process on single page

In [ ]:
cleaned_page_1 = clean_hyphenated_line_breaks(pages_text[0])
cleaned_page_1 = clean_split_words(cleaned_page_1)
cleaned_page_1 = clean_missing_spaces(cleaned_page_1)
normalized_page_1 = normalize_line_breaks(cleaned_page_1)

print(normalized_page_1[:2000])

On November 30, a research team led by University of Delaware marine scientist Craig Cary will set sail from Manzanillo, Mexico, on the 21-day expedition Extreme 2004: Exploring the Deep Frontier. They will live aboard the 274-foot research vessel Atlantis and dive to the depths in the submersible Alvin. Their mission will be to explore one of the most demanding environments on the planet: hydrothermal vents. These underwater geysers erupt at cracks in the Earth’s crust, where new seafloor is being born. The scientists will be working to learn more about biocomplexity, the complex interplay between vent organisms and their environment. Biologists, chemists, and geologists will all be working together to better understand how the strange life at the vents can function in their hostile environment of extreme temperature, toxic chemicals, high pressure, and darkness. Using tools borrowed from the Human Genome Project, the scientists will examine creatures such as the Pompeii worm, which c

In [ ]:
print("Original characters:", len(pages_text[0]))
print("Cleaned characters:", len(normalized_page_1))

print("\nRemaining split-word pattern:")
print(re.findall(r"(?m)^([A-Z])\n([a-z])", normalized_page_1))

print("\nRemaining missing-space pattern:")
print(re.findall(r"[.!?][A-Z]", normalized_page_1))

Original characters: 3721
Cleaned characters: 3674

Remaining split-word pattern:
[]

Remaining missing-space pattern:
[]


## Applying all the technquies to text to clean

In [ ]:
cleaned_pages = []

for page_number, page_text in enumerate(pages_text, start=1):
    cleaned_text = clean_hyphenated_line_breaks(page_text)
    cleaned_text = clean_split_words(cleaned_text)
    cleaned_text = clean_missing_spaces(cleaned_text)
    cleaned_text = normalize_line_breaks(cleaned_text)

    cleaned_pages.append({
        "page": page_number,
        "text": cleaned_text
    })

print("Total cleaned pages:", len(cleaned_pages))

for page in cleaned_pages:
    print(
        f"Page {page['page']}: "
        f"{len(page['text'])} characters"
    )


Total cleaned pages: 8
Page 1: 3674 characters
Page 2: 5150 characters
Page 3: 4788 characters
Page 4: 5343 characters
Page 5: 4870 characters
Page 6: 6280 characters
Page 7: 4212 characters
Page 8: 5338 characters


# Chunking
**Objective**: Divide the cleaned PDF text into smaller, overlapping chunks while preserving source-page metadata.

**Configuration**:

Chunk size: 250 words

Overlap: 50 words

Why preserve page numbers? When the retriever finds relevant evidence, we need to identify the source page so our assistant can provide citations.

In [ ]:
chunk_size = 250
overlap = 50
step = chunk_size - overlap

chunks = []

for page in cleaned_pages:
  words = page['text'].split()

  for start in range(0, len(words), step):
    chunk_words = words[start : start+chunk_size]
    chunk = {
        "chunk_id" : len(chunks),
        "page" : page['page'],
        "text" : " ".join(chunk_words)
    }

    chunks.append(chunk)

    if start+chunk_size >= len(words):
      break

print("Total Chunks : ", len(chunks))
print("\nFirst Chunk : ")
print(chunks[0])


Total Chunks :  34

First Chunk : 
{'chunk_id': 0, 'page': 1, 'text': 'On November 30, a research team led by University of Delaware marine scientist Craig Cary will set sail from Manzanillo, Mexico, on the 21-day expedition Extreme 2004: Exploring the Deep Frontier. They will live aboard the 274-foot research vessel Atlantis and dive to the depths in the submersible Alvin. Their mission will be to explore one of the most demanding environments on the planet: hydrothermal vents. These underwater geysers erupt at cracks in the Earth’s crust, where new seafloor is being born. The scientists will be working to learn more about biocomplexity, the complex interplay between vent organisms and their environment. Biologists, chemists, and geologists will all be working together to better understand how the strange life at the vents can function in their hostile environment of extreme temperature, toxic chemicals, high pressure, and darkness. Using tools borrowed from the Human Genome Project, 

### Verifying Chunks

In [ ]:
# Check chunk lengths
chunk_lengths = [len(chunk["text"].split()) for chunk in chunks]

print("Total chunks:", len(chunks))
print("Shortest chunk:", min(chunk_lengths), "words")
print("Longest chunk:", max(chunk_lengths), "words")
print("First chunk word count:", chunk_lengths[0])

Total chunks: 34
Shortest chunk: 73 words
Longest chunk: 250 words
First chunk word count: 250


In [ ]:
chunk_1_words = chunks[0]["text"].split()
chunk_2_words = chunks[1]["text"].split()

print("Last 50 words of Chunk 1 match first 50 of Chunk 2:",
      chunk_1_words[-50:] == chunk_2_words[:50])

Last 50 words of Chunk 1 match first 50 of Chunk 2: True


In [ ]:
for chunk in chunks[:5]:
    print(
        "Chunk ID:", chunk["chunk_id"],
        "| Page:", chunk["page"],
        "| Words:", len(chunk["text"].split())
    )

Chunk ID: 0 | Page: 1 | Words: 250
Chunk ID: 1 | Page: 1 | Words: 250
Chunk ID: 2 | Page: 1 | Words: 182
Chunk ID: 3 | Page: 2 | Words: 250
Chunk ID: 4 | Page: 2 | Words: 250


# Embeddings

Convert each text chunk into a numerical vector that captures its semantic meaning. These vectors will later allow us to retrieve relevant document chunks when a user asks a question.

We will use the pretrained all-MiniLM-L6-v2 model from Hugging Face through the Sentence Transformers library. It produces 384-dimensional embeddings.

In [ ]:
# sentence-transformers provides tools for loading pretrained embedding models and converting text into vectors.

!pip install -q sentence-transformers
print("sentence-trasformers successfully installed")

sentence-trasformers successfully installed


In [ ]:
# Load the pretrained model

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [ ]:
# Inspecting the model

print("Model:", model)
print("Embedding dimension:", model.get_embedding_dimension())
print("Maximum sequence length:", model.max_seq_length)

Model: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
Embedding dimension: 384
Maximum sequence length: 256


In [ ]:
tokenizer = model.tokenizer

for chunk in chunks[:5]:
    tokens = tokenizer(
        chunk["text"],
        add_special_tokens=True,
        truncation=False
    )["input_ids"]

    print(
        "Chunk ID:", chunk["chunk_id"],
        "| Page:", chunk["page"],
        "| Tokens:", len(tokens)
    )

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (358 > 256). Running this sequence through the model will result in indexing errors


Chunk ID: 0 | Page: 1 | Tokens: 358
Chunk ID: 1 | Page: 1 | Tokens: 334
Chunk ID: 2 | Page: 1 | Tokens: 244
Chunk ID: 3 | Page: 2 | Tokens: 340
Chunk ID: 4 | Page: 2 | Tokens: 320


In [ ]:
# since, token size is exceeding in some chunks. we are creating chunks again for this model without changing the original chunks for baseline.\

chunk_size = 150
overlap = 30
step = chunk_size - overlap

token_chunks = []

for page in cleaned_pages:
    text = page["text"]

    # Convert the page text into tokens
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    # Split tokens into overlapping chunks
    for start in range(0, len(token_ids), step):
        chunk_token_ids = token_ids[start : start + chunk_size]

        # Convert tokens back into readable text
        chunk_text = tokenizer.decode(
            chunk_token_ids,
            skip_special_tokens=True
        )

        token_chunks.append({
            "chunk_id": len(token_chunks),
            "page": page["page"],
            "text": chunk_text
        })

        if start + chunk_size >= len(token_ids):
            break

print("Total token chunks:", len(token_chunks))
print("First chunk:", token_chunks[0])

Total token chunks: 76
First chunk: {'chunk_id': 0, 'page': 1, 'text': 'on november 30, a research team led by university of delaware marine scientist craig cary will set sail from manzanillo, mexico, on the 21 - day expedition extreme 2004 : exploring the deep frontier. they will live aboard the 274 - foot research vessel atlantis and dive to the depths in the submersible alvin. their mission will be to explore one of the most demanding environments on the planet : hydrothermal vents. these underwater geysers erupt at cracks in the earth ’ s crust, where new seafloor is being born. the scientists will be working to learn more about biocomplexity, the complex interplay between vent organisms and their environment. biologists, chemists, and geologists will all be working together to better understand'}


In [ ]:
# verify the chunks

token_lengths = [
    len(tokenizer.encode(
        chunk["text"],
        add_special_tokens=True,
        truncation=False
    ))
    for chunk in token_chunks
]

print("Total chunks:", len(token_chunks))
print("Shortest chunk:", min(token_lengths), "tokens")
print("Longest chunk:", max(token_lengths), "tokens")
print("Chunks exceeding model limit:", sum(
    length > model.max_seq_length for length in token_lengths
))

Total chunks: 76
Shortest chunk: 47 tokens
Longest chunk: 155 tokens
Chunks exceeding model limit: 0


In [ ]:
# Generate Embedding

texts = [chunk['text'] for chunk in token_chunks]

embeddings = model.encode(
    texts,
    convert_to_numpy = True,
    show_progress_bar= True
)

print("Embedding shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding shape: (76, 384)
Data type: float32


In [ ]:
# Insoecting one embedding

print("First chunk text:")
print(token_chunks[0]["text"])

print("\nFirst chunk embedding:")
print(embeddings[0])

print("\nNumber of values:", len(embeddings[0]))

First chunk text:
on november 30, a research team led by university of delaware marine scientist craig cary will set sail from manzanillo, mexico, on the 21 - day expedition extreme 2004 : exploring the deep frontier. they will live aboard the 274 - foot research vessel atlantis and dive to the depths in the submersible alvin. their mission will be to explore one of the most demanding environments on the planet : hydrothermal vents. these underwater geysers erupt at cracks in the earth ’ s crust, where new seafloor is being born. the scientists will be working to learn more about biocomplexity, the complex interplay between vent organisms and their environment. biologists, chemists, and geologists will all be working together to better understand

First chunk embedding:
[-7.84421563e-02 -7.85095096e-02  8.94587487e-03  1.86651852e-02
  6.37034560e-03 -1.53985128e-01 -1.14731848e-01 -3.03252526e-02
 -5.07621504e-02 -1.93910711e-02 -2.92205475e-02 -3.04686446e-02
 -5.61648123e-02  7.4680

# Vector Similarity search

In [ ]:
# Create a question embedding

query = "How do hydrothermal vents support life?"
query_embedding = model.encode(
    query,
    convert_to_numpy = True
)

print("Query Embedding Size : ", query_embedding.shape)

Query Embedding Size :  (384,)


In [ ]:
# calculate cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(
    query_embedding.reshape(1, -1),
    embeddings
)

print("Similarity scores shape:", similarity_scores.shape)
print(similarity_scores)

Similarity scores shape: (1, 76)
[[ 0.45580763  0.5351331   0.4871638   0.5139786   0.5590346   0.58715665
   0.20505303  0.37401783  0.21919744  0.12151346  0.06259616  0.34115002
   0.4340938   0.21550786  0.2157261   0.43041372  0.31192046  0.3004531
   0.28453457  0.63052094  0.5637228   0.4956503   0.47765154  0.60016054
   0.3799319   0.4921143   0.666402    0.45842162  0.50628674  0.44128522
   0.59227717  0.48679864 -0.03045473 -0.00796991  0.391455    0.18652913
   0.36647314  0.3051955   0.28219038  0.40263414  0.23107909  0.32472646
   0.38627368  0.08348333  0.23669368  0.02912003  0.06987906  0.12890162
   0.11051701  0.12323404  0.20965053  0.1566857   0.14270051  0.10667449
   0.16165185  0.19285339  0.12931898  0.15738165  0.14655584  0.01848952
  -0.00602777  0.08798023  0.18790871  0.15918228  0.02853357  0.47455496
   0.45470008  0.149849    0.16367516  0.11967736  0.3693291   0.15774038
   0.15193346  0.22131442  0.4112617   0.18891963]]


In [ ]:
import numpy as np

scores = similarity_scores[0]

top_indices = np.argsort(scores)[::-1][:3]

for rank, idx in enumerate(top_indices, start=1):
    print(f"\nRank: {rank}")
    print(f"Chunk ID: {token_chunks[idx]['chunk_id']}")
    print(f"Page: {token_chunks[idx]['page']}")
    print(f"Similarity score: {scores[idx]:.4f}")
    print(f"Text: {token_chunks[idx]['text'][:500]}")
    print("-" * 50)


Rank: 1
Chunk ID: 26
Page: 4
Similarity score: 0.6664
Text: visit us at www. ocean. udel. edu / extreme2004 they can withstand a broad range of temperatures, bathe in — and in many cases — even eat, toxic chemicals, bear the weight of the ocean on their shoulders, and never see the light of day. such is the life of many of the organisms that inhabit hydrothermal vent sites! these creatures, from microscopic bacteria to towering tubeworms, thrive under some of the most demanding conditions on the planet. and what further distinguishes them from other life
--------------------------------------------------

Rank: 2
Chunk ID: 19
Page: 3
Similarity score: 0.6305
Text: . solid inner core diameter : 2, 400 km many of us are familiar with “ old faithful ” in y ellowstone national park. this famous geyser erupts several times a day. it spouts a column of water heated by volcanic rock deep within the earth ’ s crust. a hydrothermal vent is a geyser on the seafloor. it continuously gushes super

# Create a reusable retrieval function

In [ ]:
def retrieve_chunks(query, top_k=3):
    # Convert the question into an embedding
    query_embedding = model.encode(
        query,
        convert_to_numpy=True
    )

    # Compare the question with all chunk embeddings
    scores = cosine_similarity(
        query_embedding.reshape(1, -1),
        embeddings
    )[0]

    # Get the indices of the top-scoring chunks
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Return the matching chunks and their scores
    results = []

    for idx in top_indices:
        results.append({
            "chunk_id": token_chunks[idx]["chunk_id"],
            "page": token_chunks[idx]["page"],
            "text": token_chunks[idx]["text"],
            "score": float(scores[idx])
        })

    return results

In [ ]:
# Testing the function
query = "What are hydrothermal vents?"

results = retrieve_chunks(query, top_k=3)

for rank, result in enumerate(results, start=1):
    print(f"\nRank: {rank}")
    print(f"Page: {result['page']}")
    print(f"Score: {result['score']:.4f}")
    print(f"Text: {result['text'][:400]}")
    print("-" * 50)


Rank: 1
Page: 3
Score: 0.6765
Text: . solid inner core diameter : 2, 400 km many of us are familiar with “ old faithful ” in y ellowstone national park. this famous geyser erupts several times a day. it spouts a column of water heated by volcanic rock deep within the earth ’ s crust. a hydrothermal vent is a geyser on the seafloor. it continuously gushes super - hot, mineral - rich water that supports a diverse community of organism
--------------------------------------------------

Rank: 2
Page: 3
Score: 0.6552
Text: taller than you to ghost - white crabs. hydrothermal vents were discovered in 1977 in the pacific ocean. since then, they have been found in the atlantic, indian, and most recently, the arctic ocean. most occur at an average depth of about 2, 100 meters ( 7, 000 ft ) in areas of seafloor spreading along the mid - ocean ridge system — the underwater mountain chain that winds around the globe. how d
--------------------------------------------------

Rank: 3
Page: 3
Score

# FAISS - Facebook AI similarity search
It's an open-source library originally developed by Facebook AI Research for searching and clustering dense vectors efficiently.

In [ ]:
!pip install -q faiss-cpu

print("FAISS Installation Succesful")

FAISS Installation Succesful


In [ ]:
import faiss

print("FAISS version:", faiss.__version__)
print("Embedding shape:", embeddings.shape)

FAISS version: 1.15.1
Embedding shape: (76, 384)


In [ ]:
# FAISS index - We'll use IndexFlatIP, which performs exact inner-product search.

# Create an exact similarity-search index
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

print("Index created successfully!")
print("Vector dimensions:", index.d)
print("Vectors stored:", index.ntotal)

Index created successfully!
Vector dimensions: 384
Vectors stored: 0


In [ ]:
# adding our embeddings

# FAISS expects float32 vectors
embeddings_float32 = np.asarray(embeddings, dtype=np.float32)

# Add all document vectors to the index
index.add(embeddings_float32)

print("Vectors added:", index.ntotal)

Vectors added: 76


In [ ]:
def retrieve_chunks_faiss(query, top_k=3):
    query_embedding = model.encode(
        query,
        convert_to_numpy=True
    )

    query_vector = np.asarray(
        query_embedding.reshape(1, -1),
        dtype=np.float32
    )

    scores, indices = index.search(query_vector, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "chunk_id": token_chunks[idx]["chunk_id"],
            "page": token_chunks[idx]["page"],
            "text": token_chunks[idx]["text"],
            "score": float(score)
        })

    return results

In [ ]:
results = retrieve_chunks_faiss(
    "How do hydrothermal vents support life?",
    top_k=3
)

for result in results:
    print(result["page"], result["score"], result["text"][:200])

4 0.6664019227027893 visit us at www. ocean. udel. edu / extreme2004 they can withstand a broad range of temperatures, bathe in — and in many cases — even eat, toxic chemicals, bear the weight of the ocean on their should
3 0.6305208206176758 . solid inner core diameter : 2, 400 km many of us are familiar with “ old faithful ” in y ellowstone national park. this famous geyser erupts several times a day. it spouts a column of water heated b
3 0.6001604795455933 about hydrothermal vents. besides providing habitat for an intriguing community of life, these underwater geysers are believed to play an important role in the ocean ’ s temperature, chemistry, and ci


# prompt Engineering

In [ ]:

query = "How do hydrothermal vents support life?"

retrieved_chunks = retrieve_chunks_faiss(query, top_k=3)

for chunk in retrieved_chunks:
    print("Page:", chunk["page"])
    print("Chunk ID:", chunk["chunk_id"])
    print("Text:", chunk["text"][:300])
    print("-" * 50)

Page: 4
Chunk ID: 26
Text: visit us at www. ocean. udel. edu / extreme2004 they can withstand a broad range of temperatures, bathe in — and in many cases — even eat, toxic chemicals, bear the weight of the ocean on their shoulders, and never see the light of day. such is the life of many of the organisms that inhabit hydrothe
--------------------------------------------------
Page: 3
Chunk ID: 19
Text: . solid inner core diameter : 2, 400 km many of us are familiar with “ old faithful ” in y ellowstone national park. this famous geyser erupts several times a day. it spouts a column of water heated by volcanic rock deep within the earth ’ s crust. a hydrothermal vent is a geyser on the seafloor. it
--------------------------------------------------
Page: 3
Chunk ID: 23
Text: about hydrothermal vents. besides providing habitat for an intriguing community of life, these underwater geysers are believed to play an important role in the ocean ’ s temperature, chemistry, and circulation patte

In [ ]:
context = ""

for chunk in retrieved_chunks:
    context += (
        f"[Page {chunk['page']}]\n"
        f"{chunk['text']}\n\n"
    )

print(context)


[Page 4]
visit us at www. ocean. udel. edu / extreme2004 they can withstand a broad range of temperatures, bathe in — and in many cases — even eat, toxic chemicals, bear the weight of the ocean on their shoulders, and never see the light of day. such is the life of many of the organisms that inhabit hydrothermal vent sites! these creatures, from microscopic bacteria to towering tubeworms, thrive under some of the most demanding conditions on the planet. and what further distinguishes them from other life on earth is their energy source. this is the only complex ecosystem known to live on energy from chemicals from the earth rather than energy from the sun. on land and in the upper layer of the ocean, the food chain

[Page 3]
. solid inner core diameter : 2, 400 km many of us are familiar with “ old faithful ” in y ellowstone national park. this famous geyser erupts several times a day. it spouts a column of water heated by volcanic rock deep within the earth ’ s crust. a hydrothermal v

In [ ]:
prompt = f"""
You are a document-grounded question-answering assistant.

Rules:
1. Answer using only the evidence provided below.
2. Do not invent facts or citations.
3. If the evidence is insufficient, clearly say so.
4. If only part of the question is supported,
   answer that part and explain what is missing.
5. Cite the source page numbers for factual claims.

Evidence:
{context}

User question:
{query}

Write a clear, concise answer supported by the evidence.
"""

print(prompt)



You are a document-grounded question-answering assistant.

Rules:
1. Answer using only the evidence provided below.
2. Do not invent facts or citations.
3. If the evidence is insufficient, clearly say so.
4. If only part of the question is supported,
   answer that part and explain what is missing.
5. Cite the source page numbers for factual claims.

Evidence:
[Page 2]
bottom of the sub. u. s. naval historical center photograph students, can you convert the measurements into metric units? understanding where hydrothermal vents occur requires a closer look at the earth ’ s structure and the forces at work deep within the planet. the deeper you go inside the earth, the hotter it gets. scientists have calculated that the earth ’ s inner core — a solid sphere composed primarily of iron — is about 5, 500°c ( 10, 000°f ). that ’ s about the same temperature as the surface of the sun. the solid inner core is about 2, 400 kilometers ( 1, 500 mi ) in diameter. it is surrounded by a liquid oute

# API settings
We are using groq API

In [ ]:
!pip install -q groq

print("Installation Succesful")

Installation Succesful


In [ ]:
#Accessing the Secrets

from google.colab import userdata
from groq import Groq

api_key = userdata.get("Groq_API_Key")

client = Groq(api_key=api_key)

print("Groq client initialized successfully!")


Groq client initialized successfully!


In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Explain hydrothermal vents in two simple sentences."
        }
    ],
    temperature=0.2
)

print(response.choices[0].message.content)


Hydrothermal vents are openings on the seafloor where super‑hot, mineral‑rich water gushes out from beneath the Earth's crust. This hot water supports unique ecosystems that rely on chemical energy instead of sunlight.
